# 02. Repeat Order Retention

Основная retention-метрика для food delivery: доля новых пользователей, которые сделали хотя бы один повторный доставленный заказ в течение 30 дней после первого заказа. Дополнительно смотрим раннюю динамику по окнам 7/14/30 дней и сравниваем месячные когорты первого заказа.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

orders = pd.read_csv(Path('../data/fact_orders.csv'), parse_dates=['order_ts'])
orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_date'] = orders['order_ts'].dt.normalize()

In [2]:
first_orders = orders.groupby('user_id', as_index=False)['order_ts'].min().rename(columns={'order_ts': 'first_order_ts'})
first_orders['first_order_date'] = first_orders['first_order_ts'].dt.normalize()
first_orders['cohort_month'] = first_orders['first_order_ts'].dt.to_period('M').dt.to_timestamp()

repeat_orders = orders.merge(first_orders, on='user_id', how='inner')
repeat_orders = repeat_orders[repeat_orders['order_ts'] > repeat_orders['first_order_ts']].copy()
repeat_orders['days_after_first_order'] = (
    repeat_orders['order_ts'].dt.normalize() - repeat_orders['first_order_ts'].dt.normalize()
).dt.days

max_order_ts = orders['order_ts'].max()
max_order_date = max_order_ts.normalize()
first_orders.head()

,user_id,first_order_ts,first_order_date,cohort_month
0,1,2025-11-16 23:21:09.261451213,2025-11-16,2025-11-01
1,2,2025-06-30 06:48:50.752582127,2025-06-30,2025-06-01
2,5,2025-02-12 05:33:41.448714605,2025-02-12,2025-02-01
3,6,2026-02-04 17:40:32.413199027,2026-02-04,2026-02-01
4,7,2025-11-11 22:52:50.333054299,2025-11-11,2025-11-01


In [3]:
horizons = [7, 14, 30]

rows = []
for horizon in horizons:
    eligible = first_orders.loc[
        first_orders['first_order_ts'] <= max_order_ts - pd.Timedelta(days=horizon)
    ].copy()
    repeated_users = repeat_orders.loc[
        repeat_orders['user_id'].isin(eligible['user_id'])
        & repeat_orders['days_after_first_order'].between(1, horizon),
        'user_id'
    ].nunique()
    users_base = eligible['user_id'].nunique()
    rows.append({
        'window_days': horizon,
        'users_base': users_base,
        'repeat_users': repeated_users,
        'repeat_rate_pct': repeated_users / users_base * 100 if users_base else 0,
    })

repeat_by_window = pd.DataFrame(rows)
repeat_by_window


,window_days,users_base,repeat_users,repeat_rate_pct
0,7,10193,3503,34.366722
1,14,10193,5285,51.849308
2,30,10192,5840,57.299843


In [4]:
main_window = 30

eligible_30 = first_orders.loc[
    first_orders['first_order_ts'] <= max_order_ts - pd.Timedelta(days=main_window)
].copy()

repeat_users_30 = repeat_orders.loc[
    repeat_orders['user_id'].isin(eligible_30['user_id'])
    & repeat_orders['days_after_first_order'].between(1, main_window),
    'user_id'
].unique()

eligible_30['has_repeat_30d'] = eligible_30['user_id'].isin(repeat_users_30)
repeat_30_rate = eligible_30['has_repeat_30d'].mean() * 100
repeat_30_users = int(eligible_30['has_repeat_30d'].sum())
eligible_30_users = eligible_30['user_id'].nunique()

monthly_repeat_30 = (
    eligible_30
    .groupby('cohort_month', as_index=False)
    .agg(
        users_base=('user_id', 'nunique'),
        repeat_users=('has_repeat_30d', 'sum'),
    )
)
monthly_repeat_30['repeat_rate_pct'] = monthly_repeat_30['repeat_users'] / monthly_repeat_30['users_base'] * 100
monthly_repeat_30['cohort_label'] = monthly_repeat_30['cohort_month'].dt.strftime('%b %Y')

summary = pd.DataFrame({
    '\u041c\u0435\u0442\u0440\u0438\u043a\u0430': ['Repeat order rate 30D', '\u041f\u043e\u0432\u0442\u043e\u0440\u0438\u0432\u0448\u0438\u0445 \u0437\u0430\u043a\u0430\u0437', '\u0411\u0430\u0437\u0430 \u0440\u0430\u0441\u0447\u0435\u0442\u0430'],
    '\u0417\u043d\u0430\u0447\u0435\u043d\u0438\u0435': [
        f'{repeat_30_rate:.1f}%',
        f'{repeat_30_users:,}'.replace(',', ' '),
        f'{eligible_30_users:,}'.replace(',', ' '),
    ],
})
display(summary)

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.48, 0.52],
    horizontal_spacing=0.13,
)

window_labels = repeat_by_window['window_days'].map(lambda day: f'{int(day)} \u0434\u043d\u0435\u0439')
window_values = repeat_by_window['repeat_rate_pct'].round(1)
window_colors = ['#D1D5DB' if day != main_window else '#FFCC00' for day in repeat_by_window['window_days']]

fig.add_trace(
    go.Bar(
        x=window_labels,
        y=window_values,
        marker=dict(color=window_colors, line=dict(color='white', width=1.5)),
        text=[f'{value:.1f}%' for value in window_values],
        textposition='outside',
        cliponaxis=False,
        hovertemplate='%{x}<br>Repeat rate: %{y:.1f}%<extra></extra>',
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=monthly_repeat_30['cohort_label'],
        y=monthly_repeat_30['repeat_rate_pct'].round(1),
        mode='lines+markers',
        line=dict(color='#111827', width=3),
        marker=dict(size=8, color='#FFCC00', line=dict(color='#111827', width=1.5)),
        text=[f"n={int(n):,}".replace(',', ' ') for n in monthly_repeat_30['users_base']],
        hovertemplate='\u041a\u043e\u0433\u043e\u0440\u0442\u0430: %{x}<br>30D repeat rate: %{y:.1f}%<br>%{text}<extra></extra>',
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.add_annotation(
    xref='paper',
    yref='paper',
    x=0.00,
    y=1.12,
    text=f'<b>\u041f\u043e\u0432\u0442\u043e\u0440\u043d\u044b\u0435 \u0437\u0430\u043a\u0430\u0437\u044b \u043f\u043e\u0441\u043b\u0435 \u043f\u0435\u0440\u0432\u043e\u0433\u043e \u0437\u0430\u043a\u0430\u0437\u0430</b><br><span style="font-size:14px;color:#4B5563">30D repeat rate = {repeat_30_rate:.1f}% | \u0431\u0430\u0437\u0430: {eligible_30_users:,} \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u0435\u0439 \u0441 \u043f\u043e\u043b\u043d\u044b\u043c \u043e\u043a\u043d\u043e\u043c \u043d\u0430\u0431\u043b\u044e\u0434\u0435\u043d\u0438\u044f</span>'.replace(',', ' '),
    showarrow=False,
    align='left',
    font=dict(size=24, color='#111827'),
)
fig.add_annotation(
    xref='paper',
    yref='paper',
    x=0.22,
    y=0.99,
    text='<b>\u041d\u0430\u043a\u043e\u043f\u0438\u0442\u0435\u043b\u044c\u043d\u044b\u0439 repeat rate</b>',
    showarrow=False,
    font=dict(size=15, color='#374151'),
)
fig.add_annotation(
    xref='paper',
    yref='paper',
    x=0.77,
    y=0.99,
    text='<b>30D repeat rate \u043f\u043e \u043c\u0435\u0441\u044f\u0447\u043d\u044b\u043c \u043a\u043e\u0433\u043e\u0440\u0442\u0430\u043c</b>',
    showarrow=False,
    font=dict(size=15, color='#374151'),
)

fig.update_layout(
    template='plotly_white',
    width=1120,
    height=620,
    margin=dict(l=78, r=55, t=105, b=95),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#374151'),
)

fig.update_xaxes(title='', showgrid=False, zeroline=False, row=1, col=1)
fig.update_yaxes(
    title='\u0414\u043e\u043b\u044f \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u0435\u0439',
    ticksuffix='%',
    range=[0, max(15, float(window_values.max()) * 1.32)],
    gridcolor='#E5E7EB',
    zeroline=False,
    row=1,
    col=1,
)
fig.update_xaxes(title='\u041c\u0435\u0441\u044f\u0446 \u043f\u0435\u0440\u0432\u043e\u0433\u043e \u0437\u0430\u043a\u0430\u0437\u0430', showgrid=False, zeroline=False, tickangle=-25, row=1, col=2)
fig.update_yaxes(
    title='30D repeat rate',
    ticksuffix='%',
    range=[0, max(15, float(monthly_repeat_30['repeat_rate_pct'].max()) * 1.20)],
    gridcolor='#E5E7EB',
    zeroline=False,
    row=1,
    col=2,
)

fig.show()


,Метрика,Значение
0,Repeat order rate 30D,57.3%
1,Повторивших заказ,5 840
2,База расчета,10 192
